# Module 4: Evaluation

In [4]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py

--2026-07-06 18:30:11--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/04-evaluation/code/evaluation_utils.py
Loaded CA certificate '/etc/ssl/certs/ca-certificates.crt'
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.109.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3073 (3.0K) [text/plain]
Saving to: ‘evaluation_utils.py’

evaluation_utils.py 100%[===================>]   3.00K  --.-KB/s    in 0s      

2026-07-06 18:30:11 (48.7 MB/s) - ‘evaluation_utils.py’ saved [3073/3073]



In [5]:
from ingest import load_faq_data

In [6]:
documents = load_faq_data()

In [8]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

As the LLM course is newer we will use the FAQs from just this course as there are far fewer FAQs.

In [9]:
from pydantic import BaseModel

In [10]:
class Questions(BaseModel):
    questions: list[str]

In [18]:
data_generation_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

### OpenAI Config

In [12]:
import os
from dotenv import dotenv_values, load_dotenv

secrets_dir = os.path.expanduser("~/Documents/.secrets/llm-zoomcamp/")

config = {
    **dotenv_values(secrets_dir + "/.env.openai"),
}

api_key = config.get("OPENAI_API_KEY")

In [13]:
from openai import OpenAI
openai_client = OpenAI(api_key=api_key)

## 

In [14]:
import json

In [22]:
doc = documents_llm[0]

In [23]:
user_prompt = json.dumps(doc)

In [24]:
messages = [
    {"role": "developer", "content": data_generation_instructions},
    {"role": "user", "content": user_prompt}
]

In [25]:
response = openai_client.responses.parse(
        model='gpt-5.4-mini',
        input=messages,
        text_format=Questions
    )

In [26]:
result = response.output_parsed
print(result)

questions=['Can I still join the course if I found it late?', 'Is it okay to start after the course has already begun?', 'If I join now, can I still get a certificate?', 'What do I need to do to be eligible for the certificate if I start late?', 'Is the project submission deadline the thing I should watch if I want the certificate?']


Generate for all documents

In [31]:
from evaluation_utils import llm_structured, calc_price

In [28]:
result, usage = llm_structured(
    openai_client,
    data_generation_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — is it still okay to join now, or am I too late?', 'Can I enroll after the course has already started?', 'If I join late, can I still get a certificate somehow?', "What do I need to do to be eligible for the certificate if I'm starting now?", 'Are project submissions still open for new students who just discovered the course?']


In [32]:
calc_price(usage)

{'input_cost': 0.00015525,
 'output_cost': 0.00040500000000000003,
 'total_cost': 0.00056025}

In [33]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — is it still okay to join now, or am I too late?',
  'document': '74eb249bbf'},
 {'question': 'Can I enroll after the course has already started?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, can I still get a certificate somehow?',
  'document': '74eb249bbf'},
 {'question': "What do I need to do to be eligible for the certificate if I'm starting now?",
  'document': '74eb249bbf'},
 {'question': 'Are project submissions still open for new students who just discovered the course?',
  'document': '74eb249bbf'}]